# 팀원 A 새 STFPM 결과 재현 확인

팀원 A이 보낸 **학습된 모델 그대로**(`outputs/checkpoints/stfpm_model.ckpt` + 기준값 파일)를 보고서와 **같은 테스트 사진 2,400장**(정상 400 + 결함 2,000)에 돌려서, 보고서 숫자가 그대로 나오는지 확인한다. 다시 학습하지 않으니 금방 끝난다.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `stfpm_compare_bundle_v4.zip`: 공유 패키지의 `3_supervised_vs_unsupervised/` 안에 있음. 평가 스크립트와 분할표
- `STFPM_Defect_Detection.zip`: 팀원 A이 보낸 zip(33MB)

**보고서 숫자(맞춰 볼 값)**
- 전체 2,400장: AUROC 0.9612 · 재현율 0.8155 · 오탐 5.00% · TN/FP/FN/TP = 380/20/369/1631
- 게이트 적용 2,367장: AUROC 0.9632 · TN/FP/FN/TP = 377/20/340/1630

## 1. GPU 확인 (L4 권장. CPU여도 돌지만 느리다)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
for f in ("underfill_originals_4000.zip", "stfpm_compare_bundle_v4.zip", "STFPM_Defect_Detection.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
- `/content/orig`: 원본 사진
- `/content/bundle`: 우리 평가 스크립트(`eval_checkpoint.py`)와 분할표(`stride5_split.csv` = 팀원 A 방식 테스트 구성)
- `/content/stfpm_v2/STFPM_Defect_Detection`: 팀원 A 새 코드와 모델

PDF 파일 이름이 깨졌다는 경고가 나와도 상관없다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/stfpm_compare_bundle_v4.zip" "{DRIVE_DIR}/STFPM_Defect_Detection.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/stfpm_compare_bundle_v4.zip -d /content/bundle
!unzip -q -o /content/STFPM_Defect_Detection.zip -d /content/stfpm_v2
!ls /content/stfpm_v2/STFPM_Defect_Detection/outputs/checkpoints
!find /content/orig -name "*.jpg" | wc -l

## 4. anomalib 설치 ("세션 다시 시작" 안내가 뜨면 다시 시작하고 2번 셀부터)

In [ ]:
!pip install -q "anomalib==2.6.2" timm

## 5. 팀원 A 모델로 2,400장 채점

*   항목 추가
*   항목 추가


`eval_checkpoint.py`가 팀원 A 판정기(`Printer3DDefectPredictor`)에 팀원 A 모델 파일을 넣고 사진마다 점수를 매긴다. 점수 방식(layer3, 평균, 가장자리 5% 제외)과 기준값(0.121)은 모델 옆의 `model_metadata.json`에서 자동으로 읽는다. 결과는 Drive에 CSV로 저장된다.

In [ ]:
PROJ = "/content/stfpm_v2/STFPM_Defect_Detection"
OUT = f"{DRIVE_DIR}/results/stfpm_v2_teammate_ckpt_stride5_scores.csv"
!cd /content/bundle && HF_HUB_OFFLINE=0 python eval_checkpoint.py --project "{PROJ}" --ckpt-dir "{PROJ}/outputs/checkpoints" --split-csv /content/bundle/stride5_split.csv --orig-root /content/orig --out "{OUT}"

## 6. 보고서 숫자와 비교 (게이트 포함)
게이트는 팀원 A 코드의 함수(`check_filament_presence`)를 그대로 불러서, 고정 크롭 영역의 필라멘트 비율이 25% 미만인 사진을 뺀다.

In [ ]:
import sys, cv2, pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.insert(0, PROJ)
from src.data.preprocessing import check_filament_presence, crop_roi_numpy

s = pd.read_csv(OUT, encoding="utf-8-sig")
def filament(src):
    img = cv2.imread(f"/content/orig/{src.replace(chr(92), '/')}")
    return check_filament_presence(crop_roi_numpy(img, (0, 250, 1000, 1250)), min_ratio=0.25)[1]
s["filament_ratio"] = [filament(x) for x in s["source"]]

def report(d, name):
    y, p = d["label_id"], d["pred"]
    tn, fp = int(((y == 0) & (p == 0)).sum()), int(((y == 0) & (p == 1)).sum())
    fn, tp = int(((y == 1) & (p == 0)).sum()), int(((y == 1) & (p == 1)).sum())
    print(f"{name}: {len(d)}장 | AUROC {roc_auc_score(y, d['score']):.4f} | 재현율 {tp / (tp + fn):.4f} | 오탐 {fp / (fp + tn):.4f} "
          f"| 정밀도 {tp / (tp + fp):.4f} | TN/FP/FN/TP {tn}/{fp}/{fn}/{tp}")

report(s, "전체")
report(s[s["filament_ratio"] >= 0.25], "게이트 적용")
print("\n보고서 → 전체: AUROC 0.9612, 재현율 0.8155, 오탐 0.0500, 380/20/369/1631 | 게이트: AUROC 0.9632, 377/20/340/1630")

## 7. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()